# CardioIA - Fase 2: Análise Exploratória de Dados (EDA)\n## Parte 1 - Dados Numéricos (IoT)\n\n**Projeto:** CardioIA - FIAP 2026\n**Fase:** 2 - Batimentos de Dados\n**Dataset:** Heart Failure Prediction Dataset\n**Data:** 14/04/2026\n\n### Objetivos:\n1. Compreender a estrutura e características dos dados numéricos cardiovasculares\n2. Identificar padrões, correlações e anomalias\n3. Preparar os dados para modelagem de Machine Learning\n4. Realizar limpeza e tratamento de valores ausentes/inconsistentes\n5. Gerar visualizações que auxiliem na interpretação clínica dos dados

In [ ]:
# Importação de bibliotecas essenciais\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom scipy import stats\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configurações de visualização\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\nplt.rcParams['figure.figsize'] = (12, 6)\nplt.rcParams['font.size'] = 10\n\nprint('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados\n\nO dataset contém 918 registros de pacientes com 12 variáveis relacionadas à saúde cardiovascular.

In [ ]:
# Carregar o dataset do Google Drive (link público disponível em links.md)\n# URL: https://drive.google.com/file/d/16gj5NjprTpV9a2PC1y7pvYApqnnqrL1a/view?usp=drive_link\n\n# Para executar este notebook, baixe o arquivo heart.csv e coloque na pasta data/\n# ou use o link direto (necessita configuração)\n\ndf = pd.read_csv('../data/heart.csv')\n\nprint(f'Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas')\ndf.head()

## 2. Análise Estrutural\n\nVamos entender a estrutura dos dados: tipos de variáveis, valores ausentes e estatísticas descritivas.

In [ ]:
# Informações gerais do dataset\nprint('='*60)\nprint('INFORMAÇÕES GERAIS DO DATASET')\nprint('='*60)\ndf.info()

In [ ]:
# Verificar valores ausentes\nprint('\n' + '='*60)\nprint('VALORES AUSENTES')\nprint('='*60)\nmissing = df.isnull().sum()\nmissing_pct = 100 * df.isnull().sum() / len(df)\nmissing_table = pd.concat([missing, missing_pct], axis=1)\nmissing_table.columns = ['Total', 'Percentual (%)']\nprint(missing_table[missing_table['Total'] > 0])\n\nif missing_table['Total'].sum() == 0:\n    print('✓ Não há valores ausentes no dataset!')

In [ ]:
# Estatísticas descritivas das variáveis numéricas\nprint('\n' + '='*60)\nprint('ESTATÍSTICAS DESCRITIVAS - VARIÁVEIS NUMÉRICAS')\nprint('='*60)\ndf.describe().round(2)

In [ ]:
# Distribuição das variáveis categóricas\nprint('\n' + '='*60)\nprint('DISTRIBUIÇÃO - VARIÁVEIS CATEGÓRICAS')\nprint('='*60)\n\ncategorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope', 'HeartDisease']\n\nfor col in categorical_cols:\n    print(f'\n{col}:')\n    print(df[col].value_counts())\n    print(f'Proporção: {df[col].value_counts(normalize=True).round(3)}')

## 3. Análise da Variável Alvo\n\nA variável `HeartDisease` indica se o paciente possui doença cardíaca (1) ou não (0).

In [ ]:
# Distribuição da variável alvo\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Gráfico de barras\ntarget_counts = df['HeartDisease'].value_counts()\naxes[0].bar(target_counts.index, target_counts.values, color=['#2ecc71', '#e74c3c'])\naxes[0].set_xlabel('Doença Cardíaca', fontsize=12)\naxes[0].set_ylabel('Frequência', fontsize=12)\naxes[0].set_title('Distribuição da Variável Alvo', fontsize=14, fontweight='bold')\naxes[0].set_xticks([0, 1])\naxes[0].set_xticklabels(['Ausente (0)', 'Presente (1)'])\n\n# Gráfico de pizza\naxes[1].pie(target_counts.values, labels=['Ausente', 'Presente'], autopct='%1.1f%%', startangle=90, colors=['#2ecc71', '#e74c3c'])\naxes[1].set_title('Proporção de Pacientes com Doença Cardíaca', fontsize=14, fontweight='bold')\n\nplt.tight_layout()\nplt.show()\n\nprint(f'\nBalanceamento das classes:')\nprint(f'Pacientes SEM doença cardíaca: {target_counts[0]} ({100*target_counts[0]/len(df):.1f}%)')\nprint(f'Pacientes COM doença cardíaca: {target_counts[1]} ({100*target_counts[1]/len(df):.1f}%)')

## 4. Detecção de Outliers\n\nValores extremos podem indicar erros de medição ou casos raros que merecem atenção especial.

In [ ]:
# Boxplot para detecção de outliers\nnumeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']\n\nfig, axes = plt.subplots(2, 3, figsize=(16, 10))\naxes = axes.flatten()\n\nfor i, col in enumerate(numeric_cols):\n    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True)\n    axes[i].set_title(f'Boxplot - {col}', fontsize=12, fontweight='bold')\n    axes[i].set_ylabel('Valor', fontsize=10)\n    axes[i].grid(True, alpha=0.3)\n\naxes[5].axis('off')\nplt.tight_layout()\nplt.show()

## 5. Análise de Correlações\n\nIdentificar relações entre variáveis que possam influenciar o diagnóstico de doença cardíaca.

In [ ]:
# Matriz de correlação\nplt.figure(figsize=(12, 10))\n\n# Criar cópia com variáveis codificadas\ndf_encoded = df.copy()\n\n# Codificar variáveis categóricas para correlação\nfrom sklearn.preprocessing import LabelEncoder\nle = LabelEncoder()\n\nfor col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:\n    df_encoded[col] = le.fit_transform(df[col])\n\n# Calcular correlação\ncorr_matrix = df_encoded.corr()\n\n# Heatmap\nsns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})\nplt.title('Matriz de Correlação - Variáveis CardioIA', fontsize=16, fontweight='bold')\nplt.tight_layout()\nplt.show()\n\n# Correlações mais fortes com HeartDisease\nprint('\nCorrelações com HeartDisease (ordenadas):')\nprint(corr_matrix['HeartDisease'].sort_values(ascending=False))

## 6. Conclusões da Análise Exploratória\n\n### Principais Descobertas:\n\n1. **Qualidade dos Dados**: Dataset completo sem valores ausentes, facilitando análise\n2. **Balanceamento**: Classes relativamente balanceadas (importante para ML)\n3. **Outliers**: Identificados em Cholesterol (valores = 0 podem ser erros)\n4. **Correlações Fortes**: ST_Slope, ChestPainType e ExerciseAngina mostram alta correlação com HeartDisease\n5. **Próximos Passos**: \n   - Tratar valores de Cholesterol = 0\n   - Feature engineering (criar novas variáveis)\n   - Normalização/padronização para modelagem\n   - Seleção de features baseada em importância

---\n\n**Autores**: Tiago Alves Cordeiro (RM 561791), Matheus Parra (RM 561907), Otavio Custodio (RM 565606), Thiago Henrique (RM 563327), Leandro Arthur (RM 565240)\n**Repositório**: [github.com/tiagoalvescordeiro/cardio-ia-fase1](https://github.com/tiagoalvescordeiro/cardio-ia-fase1)\n**FIAP 2026** - Inteligência Artificial - Fase 2